# 01 — Worked solutions: regression

These are reproducible examples; scores can vary with the supplied synthetic sample and library versions.

## Setup and Exercise 1

The pipeline fits transformations on training data and compares its predictions with a constant median baseline.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_data_file(filename):
    for directory in (Path.cwd(), *Path.cwd().parents):
        data_path = directory / "data" / filename
        if data_path.is_file():
            return data_path
    raise FileNotFoundError(f"Could not find data/{filename}")


housing_data = pd.read_csv(find_data_file("housing.csv"))
feature_data = housing_data.drop(columns="price")
price_response = housing_data["price"]
training_features, test_features, training_response, test_response = train_test_split(
    feature_data, price_response, test_size=0.25, random_state=27
)
numeric_columns = ["area_m2", "bedrooms", "age_years", "distance_km"]
categorical_columns = ["neighborhood"]
preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        (
            "category",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_columns,
        ),
    ]
)

linear_regression_pipeline = Pipeline(
    steps=[("preprocessing", preprocessing), ("model", LinearRegression())]
)
linear_regression_pipeline.fit(training_features, training_response)
test_predictions = linear_regression_pipeline.predict(test_features)

median_baseline = DummyRegressor(strategy="median")
baseline_training_input = np.zeros((len(training_response), 1))
baseline_test_input = np.zeros((len(test_response), 1))
median_baseline.fit(baseline_training_input, training_response)
baseline_predictions = median_baseline.predict(baseline_test_input)


def summarize_regression_errors(actual_values, predicted_values):
    return {
        "MAE": mean_absolute_error(actual_values, predicted_values),
        "RMSE": mean_squared_error(actual_values, predicted_values) ** 0.5,
        "R²": r2_score(actual_values, predicted_values),
    }


pd.DataFrame(
    {
        "Linear model": summarize_regression_errors(test_response, test_predictions),
        "Median baseline": summarize_regression_errors(test_response, baseline_predictions),
    }
).T.round(2)

The linear model should generally improve on the baseline because the synthetic generator gives price structure related to the features. Use the printed values to state whether it did on this split; exact scores are not the lesson.

## Exercise 2 — Residuals

Curvature suggests a missing nonlinear pattern; changing spread suggests that error variance may depend on predicted price. Either pattern needs follow-up. Small samples make visual patterns uncertain.

In [ ]:
test_residuals = test_response.to_numpy() - test_predictions
plt.scatter(test_predictions, test_residuals, alpha=0.75)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Predicted price")
plt.ylabel("Observed minus predicted")
plt.title("Held-out residuals")
plt.show()

## Exercise 3 — Reasoning

1. RMSE squares errors before averaging, magnifying the large miss. MAE averages absolute sizes.
2. This is extrapolation. We assume the fitted relationship continues well beyond the observed area range, which available data cannot verify.
3. Neighborhood may stand in for unmeasured location, quality, or selection factors. The coefficient is a conditional association, not an intervention effect.

## Exercise 4 — Training-only CV

The complete search below selects alpha without touching the test partition.

In [ ]:
ridge_pipeline = Pipeline(
    steps=[("preprocessing", preprocessing), ("model", Ridge())]
)
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={"model__alpha": [0.1, 1.0, 10.0, 100.0]},
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=27),
    refit=True,
)
ridge_search.fit(training_features, training_response)

cv_results = pd.DataFrame(
    {
        "alpha": ridge_search.cv_results_["param_model__alpha"].astype(float),
        "CV RMSE": -ridge_search.cv_results_["mean_test_score"],
    }
).sort_values("CV RMSE")
cv_results

Repeatedly selecting by test score adapts to random quirks in that test sample, making its reported performance optimistic. Use CV for selection and the held-out set once for final assessment.

## Exercise 5 — Example report

“The model was assessed on a held-out quarter of a synthetic dataset. Test MAE was [value] price units and RMSE was [value]. Compared with the training-median baseline, errors were [lower/higher]. The residual plot showed [observation]. The sample is synthetic and does not establish causal effects or performance on real transactions.”